In [1]:
import os
os.chdir("../..")

In [2]:
import torch
from models.mlrod import MLROD_model
from models.RamanFormer import RamanFormer
from models.RamanNet import RamanNet
from models.SANet import ScaleAdaptiveNet
from models.transformer import ViT
from torch.utils.data import Dataset,DataLoader
import pickle
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score

In [3]:
class MLROD_dataset(Dataset):
  def __init__(self,path):
    """
    path is a string containing the path to the pkl dataset
    """
    super().__init__()   
    self.y, self.X = pickle.load(open(path, 'rb'))
    self.y = list(self.y) #y is a list with containing the name of the chemical corresponding to X
    self.X = list(self.X) #X is a list with each element of the list containing a 1024 time series data    

    #To remove the Unknown samples from the dataset
    i = 0
    while i<len(self.y):
      if self.y[i]==15:
        self.y.pop(i)
        self.X.pop(i)
      
      else:
        i+=1

  def __len__(self):
    return len(self.y)

  def __getitem__(self,index):
    data = torch.Tensor(self.X[index]) #of shape (1,1024)
    data = data/data.max()
    label = self.y[index]
    return data,label

In [4]:
def test_f1(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [5]:
def test_f1_RamanNet(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred,_ = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [7]:
test_set = MLROD_dataset("datasets/MLROD/MLROD_test.pkl")
test_loader = DataLoader(test_set, batch_size=32, num_workers=8,shuffle=False)
print(f"The number of elements in the test set is {len(test_loader.dataset)}")

test_set_granite_0 = MLROD_dataset("datasets/MLROD/MLROD_test_granite_0.pkl")
test_set_granite_50 = MLROD_dataset("datasets/MLROD/MLROD_test_granite_50.pkl")
test_set_gabbro_0 = MLROD_dataset("datasets/MLROD/MLROD_test_gabbro_0.pkl")
test_set_gabbro_50 = MLROD_dataset("datasets/MLROD/MLROD_test_gabbro_50.pkl")

test_loader_granite_0 = DataLoader(test_set_granite_0, batch_size=32, num_workers=8,shuffle=False)
test_loader_granite_50 = DataLoader(test_set_granite_50, batch_size=32, num_workers=8,shuffle=False)
test_loader_gabbro_0 = DataLoader(test_set_gabbro_0, batch_size=32, num_workers=8,shuffle=False)
test_loader_gabbro_50 = DataLoader(test_set_gabbro_50, batch_size=32, num_workers=8,shuffle=False)

print(f"The number of elements in the test set is {len(test_loader_granite_0.dataset)}")
print(f"The number of elements in the test set is {len(test_loader_granite_50.dataset)}")
print(f"The number of elements in the test set is {len(test_loader_gabbro_0.dataset)}")
print(f"The number of elements in the test set is {len(test_loader_gabbro_50.dataset)}")

The number of elements in the test set is 34903
The number of elements in the test set is 11028
The number of elements in the test set is 5183
The number of elements in the test set is 8952
The number of elements in the test set is 9740


In [8]:
MLROD_mlrod = MLROD_model().to(device)
MLROD_mlrod.load_state_dict(torch.load("results/trained_models/MLROD_mlrod_1_99.99_.pt"))

MLROD_RamanFormer = RamanFormer().to(device)
MLROD_RamanFormer.load_state_dict(torch.load("results/trained_models/MLROD_RamanFormer_13_99.75_.pt"))

MLROD_RamanNet = RamanNet().to(device)
MLROD_RamanNet.load_state_dict(torch.load("results/trained_models/MLROD_RamanNet_32_99.76_.pt"))

MLROD_SANet = ScaleAdaptiveNet(num_classes=15).to(device)
MLROD_SANet.load_state_dict(torch.load("results/trained_models/MLROD_SANet_8_100.0_.pt"))

MLROD_transformer = ViT(patch_size=128,p=0.1).to(device)
MLROD_transformer.load_state_dict(torch.load("results/trained_models/MLROD_transformer_34_99.69_.pt"))

<All keys matched successfully>

In [9]:
all_acc, _, _, all_f1 = test_f1(MLROD_mlrod,device,test_loader)
gn0_acc, _, _, gn0_f1 = test_f1(MLROD_mlrod,device,test_loader_granite_0)
gn50_acc, _, _, gn50_f1 = test_f1(MLROD_mlrod,device,test_loader_granite_50)
gb0_acc, _, _, gb0_f1 = test_f1(MLROD_mlrod,device,test_loader_gabbro_0)
gb50_acc, _, _, gb50_f1 = test_f1(MLROD_mlrod,device,test_loader_gabbro_50)
print(f"{round(gn0_acc,2)}\\% & {round(gn0_f1,4)} & {round(gn50_acc,2)}\\% & {round(gn50_f1,4)} & {round(gb0_acc,2)}\\% & {round(gb0_f1,4)} & {round(gb50_acc,2)}\\% & {round(gb50_f1,4)} & {round(all_acc,2)}\\% & {round(all_f1,4)} \\\\")

100%|██████████| 305/305 [00:01<00:00, 186.58it/s]

99.22\% & 0.6797 & 76.42\% & 0.3103 & 77.42\% & 0.7589 & 53.19\% & 0.2026 & 77.4\% & 0.7371 \\


In [10]:
all_acc, _, _, all_f1 = test_f1(MLROD_SANet,device,test_loader)
gn0_acc, _, _, gn0_f1 = test_f1(MLROD_SANet,device,test_loader_granite_0)
gn50_acc, _, _, gn50_f1 = test_f1(MLROD_SANet,device,test_loader_granite_50)
gb0_acc, _, _, gb0_f1 = test_f1(MLROD_SANet,device,test_loader_gabbro_0)
gb50_acc, _, _, gb50_f1 = test_f1(MLROD_SANet,device,test_loader_gabbro_50)
print(f"{round(gn0_acc,2)}\\% & {round(gn0_f1,4)} & {round(gn50_acc,2)}\\% & {round(gn50_f1,4)} & {round(gb0_acc,2)}\\% & {round(gb0_f1,4)} & {round(gb50_acc,2)}\\% & {round(gb50_f1,4)} & {round(all_acc,2)}\\% & {round(all_f1,4)} \\\\")

100%|██████████| 305/305 [00:02<00:00, 102.21it/s]

92.85\% & 0.6713 & 80.69\% & 0.3311 & 91.13\% & 0.7874 & 52.15\% & 0.1756 & 79.25\% & 0.7038 \\


In [11]:
all_acc, _, _, all_f1 = test_f1_RamanNet(MLROD_RamanNet,device,test_loader)
gn0_acc, _, _, gn0_f1 = test_f1_RamanNet(MLROD_RamanNet,device,test_loader_granite_0)
gn50_acc, _, _, gn50_f1 = test_f1_RamanNet(MLROD_RamanNet,device,test_loader_granite_50)
gb0_acc, _, _, gb0_f1 = test_f1_RamanNet(MLROD_RamanNet,device,test_loader_gabbro_0)
gb50_acc, _, _, gb50_f1 = test_f1_RamanNet(MLROD_RamanNet,device,test_loader_gabbro_50)
print(f"{round(gn0_acc,2)}\\% & {round(gn0_f1,4)} & {round(gn50_acc,2)}\\% & {round(gn50_f1,4)} & {round(gb0_acc,2)}\\% & {round(gb0_f1,4)} & {round(gb50_acc,2)}\\% & {round(gb50_f1,4)} & {round(all_acc,2)}\\% & {round(all_f1,4)} \\\\")

100%|██████████| 305/305 [00:02<00:00, 111.45it/s]

90.4\% & 0.4568 & 65.68\% & 0.2416 & 98.51\% & 0.5337 & 41.31\% & 0.167 & 75.11\% & 0.6592 \\


In [12]:
all_acc, _, _, all_f1 = test_f1(MLROD_transformer,device,test_loader)
gn0_acc, _, _, gn0_f1 = test_f1(MLROD_transformer,device,test_loader_granite_0)
gn50_acc, _, _, gn50_f1 = test_f1(MLROD_transformer,device,test_loader_granite_50)
gb0_acc, _, _, gb0_f1 = test_f1(MLROD_transformer,device,test_loader_gabbro_0)
gb50_acc, _, _, gb50_f1 = test_f1(MLROD_transformer,device,test_loader_gabbro_50)
print(f"{round(gn0_acc,2)}\\% & {round(gn0_f1,4)} & {round(gn50_acc,2)}\\% & {round(gn50_f1,4)} & {round(gb0_acc,2)}\\% & {round(gb0_f1,4)} & {round(gb50_acc,2)}\\% & {round(gb50_f1,4)} & {round(all_acc,2)}\\% & {round(all_f1,4)} \\\\")

100%|██████████| 305/305 [00:03<00:00, 98.38it/s] 

90.76\% & 0.454 & 59.12\% & 0.2212 & 93.59\% & 0.4217 & 44.98\% & 0.1602 & 74.01\% & 0.5904 \\


In [13]:
all_acc, _, _, all_f1 = test_f1(MLROD_RamanFormer,device,test_loader)
gn0_acc, _, _, gn0_f1 = test_f1(MLROD_RamanFormer,device,test_loader_granite_0)
gn50_acc, _, _, gn50_f1 = test_f1(MLROD_RamanFormer,device,test_loader_granite_50)
gb0_acc, _, _, gb0_f1 = test_f1(MLROD_RamanFormer,device,test_loader_gabbro_0)
gb50_acc, _, _, gb50_f1 = test_f1(MLROD_RamanFormer,device,test_loader_gabbro_50)
print(f"{round(gn0_acc,2)}\\% & {round(gn0_f1,4)} & {round(gn50_acc,2)}\\% & {round(gn50_f1,4)} & {round(gb0_acc,2)}\\% & {round(gb0_f1,4)} & {round(gb50_acc,2)}\\% & {round(gb50_f1,4)} & {round(all_acc,2)}\\% & {round(all_f1,4)} \\\\")

100%|██████████| 305/305 [00:02<00:00, 142.04it/s]


92.5\% & 0.5944 & 62.01\% & 0.2612 & 95.74\% & 0.4821 & 46.2\% & 0.169 & 75.88\% & 0.6759 \\
